# 🧠 FRI-BCI: High-Performance EEG Decoder & Benchmark Runner
**Hardware Acceleration: Google Colab (A100 / H100 / L4)**

This notebook executes **Phase C (Decoder Comparison: CSP vs Riemannian vs EEGNet)** across subjects S001, S002, and S003 with held-out run 12 evaluation.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mohnclizzy/FRI-BCI-ML-RL-Decoder-Personal/blob/codex/robot-learning-design/notebooks/FRI_BCI_Colab_Master.ipynb)

In [ ]:
# [Step 1] Clone repository and configure environment
import os, sys
!git clone -b codex/robot-learning-design https://github.com/Mohnclizzy/FRI-BCI-ML-RL-Decoder-Personal.git /content/repo
%cd /content/repo
!pip install -q mne scikit-learn pyriemann torch matplotlib
!nvidia-smi
print("Environment initialized successfully!")

In [ ]:
# [Step 2] Execute Phase C Benchmark: CSP vs. Riemannian vs. EEGNet
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import mne
from mne.datasets import eegbci
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.metrics import balanced_accuracy_score, f1_score
from mne.decoding import CSP
import pyriemann
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace

print("Running Phase C comparative benchmark on PyTorch / GPU...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

# Subjects for evaluation
subjects = [1, 2, 3]
results = {}

class EEGNet(nn.Module):
    def __init__(self, n_channels=64, n_samples=480, n_classes=2, F1=8, D=2, F2=16, kernel_length=64):
        super().__init__()
        self.conv1 = nn.Conv2d(1, F1, (1, kernel_length), padding=(0, kernel_length // 2), bias=False)
        self.bn1 = nn.BatchNorm2d(F1)
        self.depthwise = nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1, bias=False)
        self.bn2 = nn.BatchNorm2d(F1 * D)
        self.act1 = nn.ELU()
        self.avgpool1 = nn.AvgPool2d((1, 4))
        self.dropout1 = nn.Dropout(0.25)
        self.separable = nn.Conv2d(F1 * D, F2, (1, 16), padding=(0, 8), bias=False)
        self.bn3 = nn.BatchNorm2d(F2)
        self.act2 = nn.ELU()
        self.avgpool2 = nn.AvgPool2d((1, 8))
        self.dropout2 = nn.Dropout(0.25)
        self.flatten = nn.Flatten()
        # compute linear size dynamically
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            x = self.dropout1(self.avgpool1(self.act1(self.bn2(self.depthwise(self.bn1(self.conv1(dummy)))))))
            x = self.dropout2(self.avgpool2(self.act2(self.bn3(self.separable(x)))))
            in_features = self.flatten(x).shape[1]
        self.fc = nn.Linear(in_features, n_classes)

    def forward(self, x):
        x = self.dropout1(self.avgpool1(self.act1(self.bn2(self.depthwise(self.bn1(self.conv1(x)))))))
        x = self.dropout2(self.avgpool2(self.act2(self.bn3(self.separable(x)))))
        return self.fc(self.flatten(x))

for sub in subjects:
    print(f"
Processing Subject {sub:03d} (Runs 4, 8 -> Train; Run 12 -> Held-out Test)...")
    # Runs 4, 8 (motor imagery hands/feet) & Run 12 (held-out)
    files_train = eegbci.load_data(sub, [4, 8], update_path=True)
    files_test = eegbci.load_data(sub, [12], update_path=True)
    
    def load_epochs(files):
        raws = [mne.io.read_raw_edf(f, preload=True, verbose=False) for f in files]
        raw = mne.concatenate_raws(raws)
        eegbci.standardize(raw)
        montage = mne.channels.make_standard_montage("standard_1005")
        raw.set_montage(montage, on_missing="ignore")
        raw.filter(7., 30., fir_design="firwin", skip_by_annotation="edge", verbose=False)
        events, _ = mne.events_from_annotations(raw, verbose=False)
        # T1 = rest, T2 = motor imagery
        epochs = mne.Epochs(raw, events, dict(rest=2, hand=3), tmin=0., tmax=3.0, baseline=None, preload=True, verbose=False)
        X = epochs.get_data(copy=True) # (n_epochs, n_channels, n_times)
        y = epochs.events[:, -1] - 2   # 0 = rest, 1 = hand
        return X, y
    
    X_train, y_train = load_epochs(files_train)
    X_test, y_test = load_epochs(files_test)
    
    # 1. CSP + LDA
    csp = CSP(n_components=4, reg=None, log=True, norm_trace=False)
    X_tr_csp = csp.fit_transform(X_train, y_train)
    X_te_csp = csp.transform(X_test)
    lda_csp = LDA()
    lda_csp.fit(X_tr_csp, y_train)
    y_pred_csp = lda_csp.predict(X_te_csp)
    ba_csp = balanced_accuracy_score(y_test, y_pred_csp)
    f1_csp = f1_score(y_test, y_pred_csp, average="macro")
    false_act_csp = np.mean(y_pred_csp[y_test == 0] == 1)
    
    # 2. Riemannian + LDA
    cov = Covariances(estimator="oas")
    ts = TangentSpace(metric="riemann")
    X_tr_ts = ts.fit_transform(cov.fit_transform(X_train))
    X_te_ts = ts.transform(cov.transform(X_test))
    lda_riem = LDA()
    lda_riem.fit(X_tr_ts, y_train)
    y_pred_riem = lda_riem.predict(X_te_ts)
    ba_riem = balanced_accuracy_score(y_test, y_pred_riem)
    f1_riem = f1_score(y_test, y_pred_riem, average="macro")
    false_act_riem = np.mean(y_pred_riem[y_test == 0] == 1)
    
    # 3. EEGNet (PyTorch)
    X_tr_t = torch.tensor(X_train[:, np.newaxis, :, :], dtype=torch.float32).to(device)
    y_tr_t = torch.tensor(y_train, dtype=torch.long).to(device)
    X_te_t = torch.tensor(X_test[:, np.newaxis, :, :], dtype=torch.float32).to(device)
    
    model = EEGNet(n_channels=X_train.shape[1], n_samples=X_train.shape[2]).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss()
    model.train()
    for epoch in range(120):
        optimizer.zero_grad()
        out = model(X_tr_t)
        loss = criterion(out, y_tr_t)
        loss.backward()
        optimizer.step()
    
    model.eval()
    with torch.no_grad():
        preds_eegnet = model(X_te_t).argmax(dim=1).cpu().numpy()
    ba_eegnet = balanced_accuracy_score(y_test, preds_eegnet)
    f1_eegnet = f1_score(y_test, preds_eegnet, average="macro")
    false_act_eegnet = np.mean(preds_eegnet[y_test == 0] == 1)
    
    results[f"S{sub:03d}"] = {
        "CSP_BA": float(ba_csp), "CSP_F1": float(f1_csp), "CSP_FalseAct": float(false_act_csp),
        "Riem_BA": float(ba_riem), "Riem_F1": float(f1_riem), "Riem_FalseAct": float(false_act_riem),
        "EEGNet_BA": float(ba_eegnet), "EEGNet_F1": float(f1_eegnet), "EEGNet_FalseAct": float(false_act_eegnet)
    }
    print(f"  [CSP]       BA: {ba_csp:.3f} | Macro F1: {f1_csp:.3f} | False Act: {false_act_csp:.3f}")
    print(f"  [Riemann]   BA: {ba_riem:.3f} | Macro F1: {f1_riem:.3f} | False Act: {false_act_riem:.3f}")
    print(f"  [EEGNet]    BA: {ba_eegnet:.3f} | Macro F1: {f1_eegnet:.3f} | False Act: {false_act_eegnet:.3f}")

# Save results JSON
import json
os.makedirs("results/decoder_comparison_phase_c_2026-11-06", exist_ok=True)
with open("results/decoder_comparison_phase_c_2026-11-06/metrics.json", "w") as f:
    json.dump(results, f, indent=2)
print("
Phase C metrics saved to results/decoder_comparison_phase_c_2026-11-06/metrics.json")

In [ ]:
# [Step 3] Plot Comparative Performance
import matplotlib.pyplot as plt

subs = list(results.keys())
x = np.arange(len(subs))
width = 0.25

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Balanced Accuracy
ax1.bar(x - width, [results[s]["CSP_BA"] for s in subs], width, label="CSP + LDA", color="#4C72B0")
ax1.bar(x, [results[s]["Riem_BA"] for s in subs], width, label="Riemann + LDA", color="#55A868")
ax1.bar(x + width, [results[s]["EEGNet_BA"] for s in subs], width, label="EEGNet (PyTorch)", color="#C44E52")
ax1.set_ylabel("Balanced Accuracy")
ax1.set_title("Held-Out Run 12 Balanced Accuracy")
ax1.set_xticks(x)
ax1.set_xticklabels(subs)
ax1.set_ylim([0, 1.05])
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.5)

# False Activation
ax2.bar(x - width, [results[s]["CSP_FalseAct"] for s in subs], width, label="CSP + LDA", color="#4C72B0")
ax2.bar(x, [results[s]["Riem_FalseAct"] for s in subs], width, label="Riemann + LDA", color="#55A868")
ax2.bar(x + width, [results[s]["EEGNet_FalseAct"] for s in subs], width, label="EEGNet (PyTorch)", color="#C44E52")
ax2.set_ylabel("REST False Activation Rate (Lower is Better)")
ax2.set_title("REST False Activation Fraction")
ax2.set_xticks(x)
ax2.set_xticklabels(subs)
ax2.set_ylim([0, 1.05])
ax2.legend()
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
fig_path = "results/decoder_comparison_phase_c_2026-11-06/comparison_chart.png"
plt.savefig(fig_path, dpi=200)
plt.show()
print(f"Figure saved to {fig_path}")

In [ ]:
# [Step 4] Sync AntiHermes Drive artifacts (if mounted) & Push Results to GitHub
import os, shutil, pathlib

# 1. Copy S006-S010 decision streams from AntiHermes if Drive is mounted
drive_isaac = pathlib.Path("/content/drive/MyDrive/AntiHermes/ubuntu/Moop/FRI-BCI-ML-RL-Decoder-Personal/outputs/isaac")
if drive_isaac.exists():
    os.makedirs("outputs/isaac", exist_ok=True)
    for f in sorted(drive_isaac.glob("S0*R12-decisions.jsonl")):
        shutil.copy2(f, f"outputs/isaac/{f.name}")
        print(f"Synced from AntiHermes: outputs/isaac/{f.name}")

# 2. Configure GitHub token from Colab Secrets (GH_TOKEN or GITHUB_TOKEN)
token_path = pathlib.Path.home() / ".github_token"
if not token_path.exists():
    try:
        from google.colab import userdata
        tok = userdata.get("GH_TOKEN") or userdata.get("GITHUB_TOKEN")
        if tok:
            token_path.write_text(tok.strip())
    except Exception:
        pass

!git config --global user.name "Mohnclizzy"
!git config --global user.email "219275238+Mohnclizzy@users.noreply.github.com"
!python3 scripts/git_push_api.py "feat: sync AntiHermes decision streams and Phase C decoder results"
